In [1]:
!pip install wandb

In [2]:
from kaggle_secrets import UserSecretsClient
import wandb

# Initialize the client to access secrets
user_secrets = UserSecretsClient()

# Get the key you just stored
api_key = user_secrets.get_secret("WANDB_API_KEY") 

# Log in to wandb
wandb.login(key=api_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: saurabh200206 (saurabh200206-self) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
# ============================================
# STEP 1: Imports
# ============================================
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
import joblib

# ============================================
# STEP 2: Load and Prepare Data
# ============================================
df = pd.read_csv("/kaggle/input/map-charting-student-math-misunderstandings/train.csv")

# Combine Category + Misconception into one label string
df['full_label'] = df['Category'].astype(str) + ":" + df['Misconception'].astype(str)

# Make input text like you used before
df['input_text'] = df.apply(
    lambda row: f"Question: {row['QuestionText']} Answer: {row['MC_Answer']} Explanation: {row['StudentExplanation']}",
    axis=1
)

# Multi-label binarizer
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['full_label'].apply(lambda x: [x]))

# Save binarizer for later use
joblib.dump(mlb, "label_binarizer.joblib")

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    df['input_text'], y, test_size=0.1, random_state=42
)

# ============================================
# STEP 3: Tokenization
# ============================================
MODEL_NAME = "/kaggle/input/deberta-v3-base-offline-files/deberta-v3-base-offline"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_encodings = tokenizer(list(X_train), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(list(X_val), truncation=True, padding=True, max_length=128)

# ============================================
# STEP 4: Dataset Class
# ============================================
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

train_dataset = CustomDataset(train_encodings, y_train)
val_dataset = CustomDataset(val_encodings, y_val)

# ============================================
# STEP 5: Model with Weighted Loss
# ============================================
from torch.nn import BCEWithLogitsLoss

# Calculate class weights based on inverse frequency
class_counts = y.sum(axis=0)  # number of samples per class
total_counts = y.shape[0]
pos_weights = (total_counts - class_counts) / (class_counts + 1e-5)
pos_weights = torch.tensor(pos_weights, dtype=torch.float)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=y.shape[1],
    problem_type="multi_label_classification",
    ignore_mismatched_sizes=True
)

# ============================================
# STEP 6: Trainer with custom loss
# ============================================


from transformers import Trainer

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = BCEWithLogitsLoss(pos_weight=pos_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir="./logs"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer
)

# ============================================
# STEP 7: Train
# ============================================
trainer.train()

# ============================================
# STEP 8: Save Artifacts
# ============================================
SAVE_DIR = "./my_finetuned_model"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
joblib.dump(mlb, f"{SAVE_DIR}/label_binarizer.joblib")

print(f"✅ Training complete. Model, tokenizer, and binarizer saved at {SAVE_DIR}")


2025-09-14 13:49:26.621438: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757857766.816273      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757857766.868388      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at /kaggle/input/deberta-v3-base-offline-files/deberta-v3-base-offline and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([65]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([65, 768]

Epoch,Training Loss,Validation Loss
1,2.581600,5.317498
2,4.358000,4.532163
3,1.639000,4.434360


✅ Training complete. Model, tokenizer, and binarizer saved at ./my_finetuned_model
